# Phase 1 LoRA Training (Kaggle T4×2)

**Run this notebook on Kaggle** with GPU (T4×2). Do not run locally unless you have the same data layout.- **Input**: Dataset linked as input (e.g. `souissiyoussef/diffusion-text2image`) → `/kaggle/input/datasets/souissiyoussef/diffusion-text2image/data`
- **Output**: `results/` (checkpoints, metrics, visualizations, reports)Ensure the dataset contains `data/processed/train/`, `data/processed/val/`, and that **src/** (config, dataset, train, evaluate, visualize, mlflow_utils) is available (e.g. in the same Kaggle dataset or copied into the notebook).

##  Charger la Configuration et les Modules

### Objectif:
Importer tous les modules du projet et vérifier les chemins.

### Ce que fait cette cellule:
- Charge config.py, dataset.py, train.py
- Vérifie que les chemins sont corrects

In [ ]:
# Remove Kaggle's preinstalled xformers (broken: aoti_torch_create_device_guard / No module named 'xformers.ops')
!pip uninstall -y xformers 2>/dev/null || true

# Install mlflow (not preinstalled on Kaggle)
!pip install -q mlflow

# Inject fake xformers with valid __spec__ so diffusers can load (we use use_xformers=False in training)
import sys
import importlib.util
spec_x = importlib.util.spec_from_loader("xformers", loader=None, origin="fake")
spec_ops = importlib.util.spec_from_loader("xformers.ops", loader=None, origin="fake")
xformers_fake = importlib.util.module_from_spec(spec_x)
xformers_ops_fake = importlib.util.module_from_spec(spec_ops)
xformers_fake.ops = xformers_ops_fake
sys.modules["xformers"] = xformers_fake
sys.modules["xformers.ops"] = xformers_ops_fake
print("Fake xformers installed in sys.modules")

## Charger et Inspecter le Dataset CelebA

### Objectif:
Charger les 182,339 images du dataset CelebA.

### Ce que fait cette cellule:
- Charge les métadonnées (train et val)
- Affiche les statistiques
- Valide les chemins des images

In [ ]:
import sys
import importlib.util
from pathlib import Path

# Remove any existing xformers
for key in list(sys.modules.keys()):
    if key == "xformers" or key.startswith("xformers."):
        del sys.modules[key]

# Install fake xformers
spec_x = importlib.util.spec_from_loader("xformers", loader=None, origin="fake")
spec_ops = importlib.util.spec_from_loader("xformers.ops", loader=None, origin="fake")
xformers_fake = importlib.util.module_from_spec(spec_x)
xformers_ops_fake = importlib.util.module_from_spec(spec_ops)
xformers_fake.ops = xformers_ops_fake
sys.modules["xformers"] = xformers_fake
sys.modules["xformers.ops"] = xformers_ops_fake

# Kaggle paths
KAGGLE_DATASET = Path("/kaggle/input/datasets/souissiyoussef/diffusion-text2image")
project_root = str(KAGGLE_DATASET) if (KAGGLE_DATASET / "src" / "config.py").exists() else None

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)
    sys.path.insert(0, str(KAGGLE_DATASET / "src"))
    
print("Project root for imports:", project_root or "current dir")

# ONLY import what we need for training
from config import TrainConfig, get_processed_root, get_results_root, get_celeba_img_dir
from dataset import Text2ImageDataset
from train import run_training
from visualize import plot_training_history, plot_fid_results, plot_dataset_distribution
from mlflow_utils import ExperimentTracker

## LANCER L'ENTRAÎNEMENT LoRA 

### Objectif:
Entraîner le modèle pendant 5,000 ou 15,000 étapes selon votre configuration.

### Ce que fait cette cellule:
- Lance la boucle d'entraînement complète
- Sauvegarde les checkpoints automatiquement
- Enregistre les métriques (loss, learning rate)
- Suivi via MLflow

### ⏱️ Durée estimée: 1-3 heures

### 📊 Métriques à surveiller:
- **Loss**: Doit diminuer progressivement
- **Learning Rate**: Augmente pendant warmup, puis constant
- **ETA**: Temps estimé restant

In [ ]:
data_root = get_processed_root()
print("Data root:", data_root)
print("Exists:", data_root.exists() if data_root else False)
train_meta = Path(data_root) / "train" / "metadata.csv" if data_root else None
print("train/metadata.csv exists:", train_meta.exists() if train_meta else False)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Charger le Modèle Entraîné avec LoRA

### Objectif:
Charger le modèle avec les poids LoRA entraînés.

### Ce que fait cette cellule:
- Charge Stable Diffusion v1.5
- Charge les poids LoRA depuis le checkpoint final
- Vérifie que LoRA est correctement chargé

In [ ]:
train_dataset = Text2ImageDataset("train", data_root=get_processed_root(), validate=False)
val_dataset = Text2ImageDataset("val", data_root=get_processed_root(), validate=False)
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
if len(train_dataset) > 0:
    s = train_dataset[0]
    print(f"Sample domain: {s['domain']}, class: {s['class_name']}")

## Générer des Images de Test

### Objectif:
Générer des images avec le modèle entraîné pour évaluer la qualité.

### Ce que fait cette cellule:
- Définit des prompts de test
- Génère une image pour chaque prompt
- Sauvegarde les images générées

In [ ]:
# Kaggle T4 settings — aligned with Run 1 (checkpoint-15000, best loss=0.023538)
TrainConfig.batch_size = 1
TrainConfig.gradient_accumulation_steps = 4
TrainConfig.max_train_steps = 15000
TrainConfig.save_steps = 2500
TrainConfig.logging_steps = 100
TrainConfig.fp16 = True
TrainConfig.use_xformers = False
TrainConfig.gradient_checkpointing = True
TrainConfig.resolve_paths()
print("Output dir:", TrainConfig.output_dir)
print("Logging dir:", TrainConfig.logging_dir)

In [ ]:
# Pre-training checklist - all should show OK
import torch
checks = []
# GPU
if torch.cuda.is_available():
    checks.append(("GPU available", True, torch.cuda.get_device_name(0)))
else:
    checks.append(("GPU available", False, "No GPU"))
# Data
data_root = get_processed_root()
meta = Path(data_root) / "train" / "metadata.csv"
checks.append(("Data loads", meta.exists(), str(data_root)))
# Model dirs writable
out_dir = Path(TrainConfig.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
test_file = out_dir / ".write_test"
try:
    test_file.write_text("ok")
    test_file.unlink()
    checks.append(("Checkpoints dir writable", True, TrainConfig.output_dir))
except Exception as e:
    checks.append(("Checkpoints dir writable", False, str(e)))
# Config
checks.append(("max_train_steps == 15000", TrainConfig.max_train_steps == 15000, f"current: {TrainConfig.max_train_steps}"))

for name, ok, detail in checks:
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {name}: {detail}")
if not all(c[1] for c in checks):
    raise RuntimeError("Some checks failed. Fix before running training.")
print("All checks passed. Ready to run continuation training (25,000 steps total, ~7.8h).")

In [ ]:
try:
    pipe, exp_tracker = run_training(TrainConfig)
    summary = exp_tracker.get_summary()
    print("Final loss:", summary.get("final_loss"))
    print("Total steps:", summary.get("total_steps"))
    print("Elapsed (s):", summary.get("elapsed_seconds"))
except Exception as e:
    import traceback
    print("Training failed:", e)
    traceback.print_exc()
    raise